# 📘 Chapter 15 — Decorators and Closures
## Phase 4: Intermediate Python

---

### 🎯 What you'll learn
- Closures — functions that remember their environment
- Decorators — wrapping functions to add behavior
- `functools.wraps` for proper decoration
- Practical decorator patterns

## 15.1 — Closures
A closure is a function that remembers variables from its enclosing scope.

In [ ]:
def make_counter(start=0):
    count = start
    
    def counter():
        nonlocal count  # Modify the enclosing variable
        count += 1
        return count
    
    return counter

c1 = make_counter()
c2 = make_counter(100)

print(c1(), c1(), c1())  # 1 2 3
print(c2(), c2())         # 101 102
print(c1())               # 4 — c1 remembers its own count!

## 15.2 — Decorators — The Basics
A decorator is a function that takes a function and returns a modified version.

In [ ]:
import functools
import time

# Step by step: what a decorator actually does
def my_decorator(func):
    @functools.wraps(func)  # Preserves original function's name/docstring
    def wrapper(*args, **kwargs):
        print(f"Before calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After calling {func.__name__}")
        return result
    return wrapper

@my_decorator  # Same as: greet = my_decorator(greet)
def greet(name):
    """Greet someone."""
    print(f"Hello, {name}!")

greet("Alice")
print(f"\nFunction name: {greet.__name__}")  # Still 'greet' thanks to @wraps

## 15.3 — Practical Decorator: Timer

In [ ]:
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱️  {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(1_000_000)
print(f"Result: {result}")

## 15.4 — Decorator with Arguments

In [ ]:
def repeat(n):
    """Decorator that calls a function n times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(n):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(3)
def say_hello(name):
    print(f"Hello, {name}!")
    return f"greeted {name}"

results = say_hello("Alice")
print(f"Results: {results}")

## 15.5 — 🔥 Practical Decorators You'll Actually Use

In [ ]:
# 1. Memoization / Caching (built into functools!)
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

# Without cache: exponential time. With cache: linear!
print(f"fib(50) = {fibonacci(50)}")
print(f"Cache info: {fibonacci.cache_info()}")

# 2. Retry decorator
def retry(max_attempts=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

import random
random.seed(42)

@retry(max_attempts=5, exceptions=(ValueError,))
def unreliable():
    if random.random() < 0.7:
        raise ValueError("Random failure!")
    return "Success!"

try:
    print(unreliable())
except ValueError:
    print("All attempts failed!")

## 15.6 — Decorator: Debug Logger

In [ ]:
def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        args_str = ", ".join(repr(a) for a in args)
        kwargs_str = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
        all_args = ", ".join(filter(None, [args_str, kwargs_str]))
        print(f"📞 {func.__name__}({all_args})")
        result = func(*args, **kwargs)
        print(f"   ↪ returned {result!r}")
        return result
    return wrapper

@debug
def add(a, b):
    return a + b

@debug
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

add(3, 5)
greet("Alice", greeting="Hi")

## ✏️ Exercises

In [ ]:
# Exercise 1: Write a decorator 'validate_types' that checks
# argument types. Usage:
# @validate_types(int, int)
# def add(a, b): return a + b
# add(1, 2) → 3, add("1", 2) → TypeError
# YOUR CODE HERE


# Exercise 2: Write a decorator that limits how many times
# a function can be called. After the limit, raise an error.
# YOUR CODE HERE


# Exercise 3: Write a decorator that caches results to a JSON file
# so the cache persists across program restarts
# YOUR CODE HERE
